# Homework 10b: Modeling, Time Series and Classification

Lag and rolling features on a synthetic price series with a regime shift and five jump days,
then both tracks: a linear model forecasting next-step return, and a logistic classifier
predicting next-step direction. Time-aware split throughout, sklearn Pipelines, and metrics
appropriate to each track.

In [1]:
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

np.random.seed(7)
sns.set_theme()
plt.rcParams['figure.figsize'] = (9, 4)

## Data

Synthetic price series: a calmer, slightly positive-drift regime for the first half, a
choppier, slightly negative-drift regime for the second half, plus five random jump days.
Built this way on purpose, so the "regime shift" this project is ultimately about has a
miniature analog to practice on here.

In [2]:
n = 500
dates = pd.bdate_range('2021-01-01', periods=n)
mu = np.where(np.arange(n) < n // 2, 0.0003, -0.0001)
sigma = np.where(np.arange(n) < n // 2, 0.01, 0.015)
eps = np.random.normal(mu, sigma)
jumps = np.zeros(n)
jump_days = np.random.choice(np.arange(20, n - 20), size=5, replace=False)
jumps[jump_days] = np.random.normal(0, 0.05, size=len(jump_days))
rets = eps + jumps
price = 100 * np.exp(np.cumsum(rets))
df = pd.DataFrame({'price': price}, index=dates)
df['ret'] = df['price'].pct_change().fillna(0.0)
df['log_ret'] = np.log1p(df['ret'])
df.head()

,price,ret,log_ret
2021-01-01,101.735412,0.000000,0.000000
2021-01-04,101.292875,-0.004350,-0.004359
2021-01-05,101.356527,0.000628,0.000628
2021-01-06,101.800950,0.004385,0.004375
2021-01-07,101.031283,-0.007561,-0.007589


## Features

Four features, all shifted by at least one day so nothing at time t uses information only
available at or after t (the leakage rule the assignment calls out):

- `lag_1`: yesterday's return.
- `roll_mean_5`: 5-day rolling mean return, itself shifted one more day.
- `roll_std_20`: 20-day rolling volatility, shifted one more day.
- `momentum_10`: cumulative return over the past 10 days, shifted one more day.

In [3]:
df['lag_1'] = df['ret'].shift(1)
df['roll_mean_5'] = df['ret'].rolling(5).mean().shift(1)
df['roll_std_20'] = df['ret'].rolling(20).std().shift(1)
df['momentum_10'] = df['ret'].rolling(10).sum().shift(1)

df['y_next_ret'] = df['ret'].shift(-1)
df['y_up'] = (df['y_next_ret'] > 0).astype(int)

df_feat = df.dropna().copy()
print('Rows after dropping NaN from rolling windows and the shifted target:', len(df_feat))
df_feat.head()

Rows after dropping NaN from rolling windows and the shifted target: 479


,price,ret,log_ret,lag_1,roll_mean_5,roll_std_20,momentum_10,y_next_ret,y_up
2021-01-29,100.198878,0.016949,0.016807,-0.014854,-0.003707,0.007370,-0.024953,0.001845,1
2021-02-01,100.383751,0.001845,0.001843,0.016949,0.002509,0.008455,-0.002068,-0.003565,0
2021-02-02,100.025880,-0.003565,-0.003571,0.001845,0.001706,0.008429,0.001192,0.020804,1
2021-02-03,102.106835,0.020804,0.020591,-0.003565,0.000685,0.008453,-0.007740,-0.000154,0
2021-02-04,102.091126,-0.000154,-0.000154,0.020804,0.004236,0.009675,0.015375,-0.014106,0


## Split
Most recent 20% held out as test, no shuffling, since this is time-ordered data.

In [4]:
features = ['lag_1', 'roll_mean_5', 'roll_std_20', 'momentum_10']
cut = int(len(df_feat) * 0.8)
train, test = df_feat.iloc[:cut], df_feat.iloc[cut:]

X_tr, X_te = train[features], test[features]
y_tr_reg, y_te_reg = train['y_next_ret'], test['y_next_ret']
y_tr_clf, y_te_clf = train['y_up'], test['y_up']
print('Train rows:', len(train), '| Test rows:', len(test))

Train rows: 383 | Test rows: 96


## Track 1: Forecasting Next-Step Return

In [5]:
reg = Pipeline([('scaler', StandardScaler()), ('linreg', LinearRegression())])
reg.fit(X_tr, y_tr_reg)
pred_reg = reg.predict(X_te)

mae = mean_absolute_error(y_te_reg, pred_reg)
rmse = mean_squared_error(y_te_reg, pred_reg) ** 0.5
print(f'MAE:  {mae:.5f}')
print(f'RMSE: {rmse:.5f}')
print(f'Naive (predict 0) RMSE: {(y_te_reg ** 2).mean() ** 0.5:.5f}')

fig, ax = plt.subplots()
ax.plot(test.index, y_te_reg.values, label='actual next-step return', alpha=0.7)
ax.plot(test.index, pred_reg, label='predicted', alpha=0.7)
ax.axhline(0, color='black', linewidth=0.6)
ax.legend()
ax.set_title('Forecast vs actual next-step return')
fig.tight_layout()
fig.savefig('forecast_vs_actual.png', dpi=110)
plt.close(fig)
print('Saved forecast_vs_actual.png')

MAE:  0.01152
RMSE: 0.01447
Naive (predict 0) RMSE: 0.01460


Saved forecast_vs_actual.png


## Track 2: Classifying Next-Step Direction

In [6]:
clf = Pipeline([('scaler', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
clf.fit(X_tr, y_tr_clf)
pred_clf = clf.predict(X_te)

acc = accuracy_score(y_te_clf, pred_clf)
prec = precision_score(y_te_clf, pred_clf, zero_division=0)
rec = recall_score(y_te_clf, pred_clf, zero_division=0)
f1 = f1_score(y_te_clf, pred_clf, zero_division=0)
print(f'Accuracy:  {acc:.3f}')
print(f'Precision: {prec:.3f}')
print(f'Recall:    {rec:.3f}')
print(f'F1:        {f1:.3f}')
print(f'Base rate (share of up-days in test): {y_te_clf.mean():.3f}')

cm = confusion_matrix(y_te_clf, pred_clf)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues')
ax.set_xlabel('predicted'); ax.set_ylabel('actual')
ax.set_title('Confusion matrix')
fig.tight_layout()
fig.savefig('confusion_matrix.png', dpi=110)
plt.close(fig)
print('Saved confusion_matrix.png')

Accuracy:  0.573
Precision: 0.545
Recall:    0.279
F1:        0.369
Base rate (share of up-days in test): 0.448
Saved confusion_matrix.png


## Interpretation

**What worked.** The classifier beats a coin flip (57.3% accuracy against a 44.8% base rate
of up-days in the test set, so a majority-class baseline of always predicting "down" would
score 55.2%). It's a real but small edge, not a strong signal, precision is 0.545 and recall
is only 0.279, so when it does call an "up" day it's right a bit more often than not, but it
misses most of the actual up days entirely.

**What failed.** The regression track barely beats a naive always-predict-zero baseline: RMSE
0.01447 versus 0.01460 for predicting no change at all. Next-step return is close to
unpredictable from these four features alone, which is the expected, honest result for daily
return forecasting and not a bug in the pipeline.

**Where assumptions might break.** All four features are lagged by at least one day, so there's
no lookahead leakage into the target, but the regime shift baked into this data (calmer/positive
in the first half, choppier/negative in the second) means the train and test statistics aren't
drawn from the same distribution: the model trains mostly on the calmer regime (train is the
first 80% of a series where the shift happens at the midpoint) and gets evaluated more on the
choppier one. That's a realistic risk for this kind of model, not just a synthetic-data quirk:
a classifier trained on a calm-market period and deployed once volatility regimes change is
exactly the failure mode this course's project is designed to catch and flag.

**How this would extend.** More features tied to the regime itself (a rolling volatility ratio,
or a HMM-derived state label like the ones stage 10b's own project instructions point toward)
would likely help more than adding still more lag/rolling variants of the same return series,
since the actual structure here is a regime shift, not a stronger autocorrelation signal
within either regime.